# E8 optional live Qwen demo

Run setup, cached fallback, and reviewed pair cells before presenting. The cached summary appears even if the GPU is unavailable. Use an NVIDIA L4 for the optional live cell. Keep the notebook private while model outputs are visible; inspect all four responses before showing them in a presentation. The input pair is from the documented AI-assisted development review, not an independently human-reviewed output example. No live output is written to Drive or Git.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys

drive.mount('/content/drive')
SHARED = Path('/content/drive/MyDrive/CMPE253-Project').resolve()
assert SHARED.is_dir()
REPO = Path('/content/cmpe253-llm-demographic-bias-audit')
URL = 'https://github.com/SJSUF25/cmpe253-llm-demographic-bias-audit.git'
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', URL, str(REPO)], check=True)
assert not subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['make', 'setup-colab'], cwd=REPO, check=True)
sys.path.insert(0, str(REPO / 'src'))
from demo.live import cached_fallback, reviewed_development_pair
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print('Ready:', COMMIT)


## Cached fallback: run this before the live cell

This verified aggregate view needs no model inference. If the optional live cell fails or stalls, interrupt it and rerun this cell. No new runtime or dataset build is required.


In [ ]:
bundle = cached_fallback(SHARED)
def show_cached():
    fairness = bundle['sections']['fairness']
    print('CACHED OFFLINE RESULTS — source-hashed bundle')
    print('Sections:', ', '.join(sorted(bundle['sections'])))
    for row in fairness['bbq_accuracy_gate_components']:
        print(row['model'], 'BBQ accuracy change:', round(row['accuracy_change'] * 100, 2), 'percentage points;', 'passes:', row['accuracy_component_passes'])
    for mode in ('canary_base', 'canary_adapter'):
        if mode in bundle['sections']:
            print(mode, 'controlled synthetic canary results available in dashboard')
    print('Human CAB and helpfulness ratings remain pending. Open cached dashboard for the eight-tab view.')
show_cached()


## Frozen CAB identity pair

This pair was included in the 85-item AI-assisted data review. The code verifies the structural manifest hash and that only the explicit gender identity word changes. It uses development data, so it does not open sealed test prompts.


In [ ]:
female, male = reviewed_development_pair(SHARED)
print('Scenario:', female['scenario_id'], 'split:', female['split'])
print('Female prompt:', female['prompt_text'])
print('Male prompt:', male['prompt_text'])


## Optional live Qwen generation

Run only when ready for the presentation. It loads the pinned FP16 M1 revision on L4 and generates female/male responses with baseline and fairness instructions using the locked greedy settings. The four outputs appear only in this notebook session. Read them before screen sharing; they are single examples, not the study estimate. If loading or generation fails, use the cached cell above immediately.


In [ ]:
GPU = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
assert GPU == 'NVIDIA L4', f'Use L4 for the pinned live comparison; found {GPU}'
from inference.adapter import load_model
model = load_model('M1')
assert model.spec.key == 'M1'
for condition in ('baseline', 'mitigation'):
    for row in (female, male):
        result = model.generate(row['prompt_text'], condition)
        print('\n', condition.upper(), row['group'].upper(), '—', result['output_token_count'], 'tokens;', result['finish_reason'])
        print(result['normalized_output'])
print('Live outputs were displayed privately and not saved. Inspect them before presenting.')
